In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
from part_1_RAG.ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [7]:
from openai import OpenAI
from toyaikit.llm import OpenAIClient

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # required by the SDK but ignored by Ollama
)

In [6]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [9]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=ollama_client
)

In [11]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [12]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

In [13]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [15]:
df_agent.head()

,question,answer_agent,answer_orig,tool_calls,cost,document
0,Can I take this course at my own pace and stil...,"No — for this course, you can follow it in sel...","No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001202,69d122f12e
1,Is a certificate available if I complete the c...,No — certificates are only available if you co...,"No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001157,69d122f12e
2,Do self-paced learners get any certificate for...,No — self-paced learners do not get a certific...,"No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001120,69d122f12e
3,Why are certificates not issued for the self-p...,Certificates aren’t issued for the self-paced ...,"No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001275,69d122f12e
4,Is peer review of capstone projects required i...,Yes — peer review of capstone projects is requ...,"No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001331,69d122f12e


In [22]:
df_agent.head()

,question,answer_agent,answer_orig,tool_calls,cost,document
0,Can I take this course at my own pace and stil...,"No — for this course, you can follow it in sel...","No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001202,69d122f12e
1,Is a certificate available if I complete the c...,No — certificates are only available if you co...,"No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001157,69d122f12e
2,Do self-paced learners get any certificate for...,No — self-paced learners do not get a certific...,"No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001120,69d122f12e
3,Why are certificates not issued for the self-p...,Certificates aren’t issued for the self-paced ...,"No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001275,69d122f12e
4,Is peer review of capstone projects required i...,Yes — peer review of capstone projects is requ...,"No, you can only get a certificate if you fini...","[{""name"": ""search"", ""arguments"": ""{\""query\"":\...",0.001331,69d122f12e


In [23]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [24]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [26]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="qwen3.5:9b"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        ollama_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [27]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

In [ ]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [30]:
df_agent_eval = pd.read_csv("data/agent-evaluations.csv")
df_agent_eval.head()

,question,document,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,Can I take this course at my own pace and stil...,69d122f12e,good,The agent's answer matches the ground truth. I...,good,The single search query is relevant to the use...
1,Is a certificate available if I complete the c...,69d122f12e,good,The agent’s answer matches the ground truth. I...,good,The single search query was relevant and inclu...
2,Do self-paced learners get any certificate for...,69d122f12e,good,The agent's answer matches the ground truth. I...,good,The single search query was relevant and inclu...
3,Why are certificates not issued for the self-p...,69d122f12e,good,The agent answer matches the ground truth. It ...,good,The tool call was reasonable and relevant. The...
4,Is peer review of capstone projects required i...,69d122f12e,bad,The agent’s answer is not aligned with the gro...,good,The search query was relevant and included imp...


In [29]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    45
bad      5
Name: count, dtype: int64

In [31]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    49
bad      1
Name: count, dtype: int64